# Process DeepSeek concept candidates
Merges prompt families and applies the existing CLAP text-space filters. This stage does not use WAV files; audio-grounded filtering occurs during CBM training.

In [ ]:
from pathlib import Path
import concept_pipeline

dataset = 'cremad'  # esc50 | urbansound8k | cremad
device = 'cuda'
max_len = 30
class_sim_cutoff = 0.85
other_sim_cutoff = 0.90


In [ ]:
init_dir = Path('data/concept_sets/deepseek_init')
prompt_types = ['important', 'superclass']
if dataset != 'cremad':
    prompt_types.append('around')
paths = [init_dir / f'deepseek_{dataset}_{kind}.json' for kind in prompt_types]
missing = [str(path) for path in paths if not path.exists()]
assert not missing, f'Generate these files first: {missing}'
paths


In [ ]:
prompt_dicts = [concept_pipeline.load_json(str(path)) for path in paths]
classes = concept_pipeline.load_classes(dataset)
candidates = concept_pipeline.merge_prompt_dicts(prompt_dicts)
print(f'Raw unique candidates: {len(candidates)}')
concepts = concept_pipeline.filter_concepts(
    candidates, classes, max_len=max_len, class_sim_cutoff=class_sim_cutoff,
    other_sim_cutoff=other_sim_cutoff, device=device, print_prob=0.05,
)
concepts = concept_pipeline.dedupe_case_insensitive(concepts)
save_path = f'data/concept_sets/{dataset}_filtered_deepseek.txt'
concept_pipeline.save_concept_text(save_path, concepts)
len(concepts), concepts[:30], save_path


In [ ]:
# Review every retained concept before CBM training.
for concept in concepts:
    print(concept)
